# Task 3 – CNN Image Classification on CIFAR-10: ConvNet architecture experiments
**Osama Yehya Ghazal · NIM 26/591650/PPA/07466 · KKPM Kelas B252 · Magister Ilmu Komputer UGM**

This notebook is based on the course notebook `colab_img_clf.ipynb` and reproduces every result in my report. What I changed:
1. **Bug fix in `myData`.** The original slices `[0:1024]` (the red channel) for all three channels, so the model only sees a red image. The fix uses `[1024:2048]` for green and `[2048:3072]` for blue. The `red_only` option reproduces the original behaviour.
2. **Configurable model `ConvNetX`.** Filters, kernel size, padding, convolutions per block, pooling type (max, average or stride-2 convolution), BatchNorm, dropout, GAP or flatten head, and sigmoid or softmax output. With the default settings it is the original ConvNet (same 40,810 parameters).
3. **Training for `EPOCHS` epochs** with the original optimizer, learning rate and batch size. The checkpoint with the lowest validation loss is kept (the early-stopping idea from the lecture), and the test set is only used for that checkpoint.
4. **Correct, fast testing.** Evaluation is batched and uses `argmax(dim=1)`; the original `argmax()` without `dim` only works with batch size 1.
5. **One loop for all experiments.** `EXPERIMENTS` lists every run in the report, including the red-only check, the extra seeds and the run with augmentation and a cosine learning-rate schedule. It saves a results table and the plots.

> Runtime: GPU (Runtime → Change runtime type → T4 GPU). One 10-epoch run takes roughly 5 minutes on a T4, and the full list takes about 2–3 hours. Comment out runs you do not need, or set `SMOKE = True` for a quick check that the code works.

In [ ]:
# ---- 0. setup -------------------------------------------------------------
import os, time, json, pickle
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import matplotlib.pyplot as plt

try:                                   # Colab: mount Google Drive
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

# >>> EDIT THIS: folder that contains data_batch_1..5 and test_batch <<<
root_data = "/content/drive/My Drive/img_clf/cifar-10-batches-py/"

EPOCHS   = 10       # original notebook: 1 epoch
SMOKE    = False    # True = tiny quick run to check the code works
device   = torch.device("cuda" if torch.cuda.is_available() else "cpu")
save_root = os.path.join(os.getcwd(), "runs")
os.makedirs(save_root, exist_ok=True)
CLASSES = ['airplane','automobile','bird','cat','deer','dog','frog','horse','ship','truck']
print(device)

## 1. Data (same files and split as the original: train = batch 1,3,5 · val = batch 2,4 · test = test_batch)

In [ ]:
def unpickle(file):
    with open(file, 'rb') as fo:
        return pickle.load(fo, encoding='bytes')

def one_hot(lbl):
    a = np.zeros(10); a[lbl] = 1
    return a

class myData(Dataset):
    def __init__(self, root_data, batch_files, fix_rgb=True, limit=None):   # fix_rgb=False = original bug
        labels, data = [], []
        for bf in batch_files:
            d = unpickle(root_data + bf); k = list(d.keys())
            labels.append(d[k[1]]); data.append(d[k[2]])
        self.dataset_label = np.concatenate(labels, axis=0)[:limit]
        self.dataset_data  = np.concatenate(data, axis=0)[:limit]
        self.fix_rgb = fix_rgb

    def __len__(self):
        return len(self.dataset_label)

    def __getitem__(self, index):
        row = self.dataset_data[index]
        if self.fix_rgb:   # FIX: R = [0:1024], G = [1024:2048], B = [2048:3072]
            img = row.reshape(3, 32, 32)                       # already C x H x W
        else:              # original notebook: all three channels = red [0:1024]
            img = np.repeat(row[0:1024].reshape(1, 32, 32), 3, axis=0)
        return {'label': one_hot(self.dataset_label[index]),
                'cls': int(self.dataset_label[index]),
                'image': img.astype(np.float32)}

lim = 500 if SMOKE else None
_sets = {}
def datasets(red_only=False):
    # train / val / test sets with the fixed loader, or with the original red-only bug
    if red_only not in _sets:
        _sets[red_only] = [myData(root_data, files, fix_rgb=not red_only, limit=lim) for files in
                           (['data_batch_1', 'data_batch_3', 'data_batch_5'], ['data_batch_2', 'data_batch_4'], ['test_batch'])]
    return _sets[red_only]

print([len(d) for d in datasets()])

## 2. Models: original ConvNet + configurable ConvNetX

In [ ]:
class ConvNet(nn.Module):            # original, unchanged
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 16, 3); self.conv2 = nn.Conv2d(16, 32, 3); self.conv3 = nn.Conv2d(32, 64, 3)
        self.fc1 = nn.Linear(64, 128); self.fc2 = nn.Linear(128, 64); self.fc3 = nn.Linear(64, 10)
        self.flatten = nn.Flatten(); self.pool = nn.MaxPool2d(2); self.globalpool = nn.AdaptiveAvgPool2d(1)
        self.relu = nn.ReLU(); self.sigm = nn.Sigmoid()
    def forward(self, x):
        x = x / 255
        x = self.pool(self.relu(self.conv1(x)))
        x = self.pool(self.relu(self.conv2(x)))
        x = self.pool(self.relu(self.conv3(x)))
        x = self.flatten(self.globalpool(x))
        x = self.relu(self.fc1(x)); x = self.relu(self.fc2(x))
        return self.sigm(self.fc3(x))

class ConvNetX(nn.Module):
    # defaults = original ConvNet; pool: 'max' | 'avg' | 'stride'; out='softmax' returns logits (use cross-entropy)
    def __init__(self, chans=(16,32,64), k=3, pad=0, convs=1, bn=False, pool='max',
                 head='gap', fc=(128,64), dropout=0.0, out='sigmoid'):
        super().__init__()
        layers, c_in, s = [], 3, 32
        for c in chans:
            for j in range(convs):
                st = 2 if (pool == 'stride' and j == convs - 1) else 1   # 'stride': last conv of a block downsamples
                layers.append(nn.Conv2d(c_in, c, k, padding=pad, stride=st)); s = (s + 2*pad - k) // st + 1
                if bn: layers.append(nn.BatchNorm2d(c))
                layers.append(nn.ReLU()); c_in = c
            if pool in ('max', 'avg'):
                layers.append(nn.MaxPool2d(2) if pool == 'max' else nn.AvgPool2d(2)); s //= 2
            assert s >= 1, 'feature map vanished: add padding or remove a block'
        self.features = nn.Sequential(*layers)
        self.gpool = nn.AdaptiveAvgPool2d(1) if head == 'gap' else nn.Identity()
        d_in = c_in if head == 'gap' else c_in * s * s
        cls = [nn.Flatten()]
        for d in fc:
            if dropout > 0: cls.append(nn.Dropout(dropout))
            cls += [nn.Linear(d_in, d), nn.ReLU()]; d_in = d
        if dropout > 0: cls.append(nn.Dropout(dropout))
        cls.append(nn.Linear(d_in, 10))
        self.classifier = nn.Sequential(*cls)
        self.out = out
    def forward(self, x):
        x = self.classifier(self.gpool(self.features(x / 255)))
        return torch.sigmoid(x) if self.out == 'sigmoid' else x

# name -> model config ('orig' = original class) + training options. Same runs and names as in the report.
E = lambda model, **train: dict(model=model, **train)
EXPERIMENTS = {
    # notebook model as given, and the data check
    'asgiven_1ep':    E('orig', red_only=True, epochs=1),       # exactly as given: red only, 1 epoch
    'orig_rgb':       E('orig'),                                 # original model, fixed loader, 10 epochs
    'sigmoid_bn':     E(dict(bn=True)),                          # sigmoid output + BatchNorm
    'ref_gray':       E(dict(out='softmax'), red_only=True),     # reference with the loader bug
    # reference (softmax output) and single changes
    'ref_softmax':    E(dict(out='softmax')),
    'width_x2':       E(dict(chans=(32,64,128), out='softmax')),
    'width_x4':       E(dict(chans=(64,128,256), out='softmax')),
    'depth_2':        E(dict(chans=(16,32), out='softmax')),
    'depth_4_pad':    E(dict(chans=(16,32,64,128), pad=1, out='softmax')),
    'vgg_2conv':      E(dict(pad=1, convs=2, out='softmax')),
    'pad_same':       E(dict(pad=1, out='softmax')),
    'kernel_5':       E(dict(k=5, pad=2, out='softmax')),
    'avg_pool':       E(dict(pool='avg', out='softmax')),
    'stride_conv':    E(dict(pool='stride', out='softmax')),
    'batchnorm':      E(dict(bn=True, out='softmax')),
    'dropout':        E(dict(dropout=0.3, out='softmax')),
    'flatten_head':   E(dict(head='flatten', out='softmax')),
    # follow-up: BatchNorm in the two deeper networks
    'depth_4_pad_bn': E(dict(chans=(16,32,64,128), pad=1, bn=True, out='softmax')),
    'vgg_2conv_bn':   E(dict(pad=1, convs=2, bn=True, out='softmax')),
    # combined model (chosen before the runs)
    'combo_A':        E(dict(chans=(32,64,128), pad=1, bn=True, out='softmax')),
    # training improvements: augmentation + cosine learning rate, 15 epochs
    'aug_combo_A':    E(dict(chans=(32,64,128), pad=1, bn=True, out='softmax'), aug=True, cosine=True, epochs=15),
    'aug_depth_4_pad_bn': E(dict(chans=(16,32,64,128), pad=1, bn=True, out='softmax'), aug=True, cosine=True, epochs=15),
}
# extra seeds for the robustness check (seed 0 is always run)
SEEDS = {'asgiven_1ep': [0, 1, 2, 3, 4], 'ref_softmax': [0, 1, 2], 'batchnorm': [0, 1, 2], 'combo_A': [0, 1, 2],
         'depth_4_pad_bn': [0, 1, 2]}
if SMOKE:   # a few runs that touch every code path
    EXPERIMENTS = {k: EXPERIMENTS[k] for k in ['asgiven_1ep', 'ref_gray', 'stride_conv', 'aug_combo_A']}
    SEEDS = {'asgiven_1ep': [0, 1]}

def build(cfg):
    return ConvNet() if cfg == 'orig' else ConvNetX(**cfg)

for n, e in EXPERIMENTS.items():
    m = build(e['model']); print(f"{n:15s} params = {sum(p.numel() for p in m.parameters()):,}")

## 3. Train / validate / test (same optimizer, lr and batch size as the original)

In [ ]:
def loss_fn(out, batch, kind):
    if kind == 'sigmoid':
        return F.binary_cross_entropy(out, batch['label'].to(device, dtype=torch.float), reduction='mean')
    return F.cross_entropy(out, batch['cls'].to(device))

def augment(x):
    # random 32x32 crop from a 4-pixel zero-padded image + random horizontal flip (per image)
    n = x.shape[0]; xp = F.pad(x, (4, 4, 4, 4))
    ox, oy = torch.randint(0, 9, (2, n))
    x = torch.stack([xp[i, :, oy[i]:oy[i] + 32, ox[i]:ox[i] + 32] for i in range(n)])
    flip = torch.rand(n, device=x.device) < 0.5
    x[flip] = x[flip].flip(3)
    return x

def run_epoch(model, loader, kind, optimizer=None, aug=False):
    train = optimizer is not None
    model.train() if train else model.eval()
    tot, cor, n = 0.0, 0, 0
    with torch.set_grad_enabled(train):
        for batch in loader:
            x = batch['image'].to(device, dtype=torch.float)
            if train and aug:
                x = augment(x)
            out = model(x)
            loss = loss_fn(out, batch, kind)
            if train:
                optimizer.zero_grad(); loss.backward(); optimizer.step()
            y = batch['cls'].to(device)
            tot += loss.item() * len(y); cor += (out.argmax(dim=1) == y).sum().item(); n += len(y)
    return tot / n, cor / n

@torch.no_grad()
def test(model, loader):
    model.eval(); preds, ys = [], []
    for batch in loader:
        preds.append(model(batch['image'].to(device, dtype=torch.float)).argmax(dim=1).cpu())
        ys.append(batch['cls'])
    p, y = torch.cat(preds).numpy(), torch.cat(ys).numpy()
    cm = np.zeros((10, 10), int); np.add.at(cm, (y, p), 1)
    prec = np.diag(cm) / np.maximum(cm.sum(0), 1); rec = np.diag(cm) / np.maximum(cm.sum(1), 1)
    f1 = 2 * prec * rec / np.maximum(prec + rec, 1e-12)
    return (p == y).mean(), f1.mean(), rec, cm

def loaders(red_only=False):
    tr, va, te = datasets(red_only)
    return (DataLoader(tr, batch_size=10, shuffle=True, num_workers=2),
            DataLoader(va, batch_size=500, shuffle=False, num_workers=2),
            DataLoader(te, batch_size=500, shuffle=False, num_workers=2))

## 4. Run all experiments

In [ ]:
results, histories = [], {}
for name, e in EXPERIMENTS.items():
    for seed in SEEDS.get(name, [0]):
        run = name if seed == 0 else f'{name}_s{seed}'
        torch.manual_seed(seed); np.random.seed(seed)
        model = build(e['model']).to(device)
        kind = 'sigmoid' if e['model'] == 'orig' else e['model'].get('out', 'sigmoid')
        epochs = 1 if SMOKE else e.get('epochs', EPOCHS)
        dl_train, dl_val, dl_test = loaders(e.get('red_only', False))
        optima = torch.optim.SGD(model.parameters(), lr=0.0025, momentum=0.9)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(optima, T_max=epochs) if e.get('cosine') else None
        run_dir = os.path.join(save_root, run); os.makedirs(run_dir, exist_ok=True)
        hist, best = [], (1e9, 0)
        for ep in range(epochs):
            t0 = time.time()
            tr_loss, tr_acc = run_epoch(model, dl_train, kind, optima, aug=e.get('aug', False))
            if sched is not None: sched.step()
            va_loss, va_acc = run_epoch(model, dl_val, kind)
            hist.append(dict(epoch=ep + 1, train_loss=tr_loss, train_acc=tr_acc, val_loss=va_loss,
                             val_acc=va_acc, time_s=time.time() - t0))
            if va_loss < best[0]:   # keep the checkpoint with the lowest validation loss
                best = (va_loss, ep + 1); torch.save(model.state_dict(), os.path.join(run_dir, 'model.pth'))
            print(f"{run} ep {ep+1:02d} train_loss {tr_loss:.4f} train_acc {tr_acc:.3f} val_loss {va_loss:.4f} val_acc {va_acc:.3f}")
        model.load_state_dict(torch.load(os.path.join(run_dir, 'model.pth')))
        acc, f1, per_cls, cm = test(model, dl_test)   # the test set is used once, on the selected checkpoint
        pd.DataFrame(hist).to_csv(os.path.join(run_dir, 'trainval_log.csv'), index=False)
        histories[run] = pd.DataFrame(hist)
        results.append(dict(name=run, experiment=name, seed=seed, params=sum(p.numel() for p in model.parameters()),
                            best_epoch=best[1], val_acc=hist[best[1] - 1]['val_acc'], test_acc=acc, macro_f1=f1,
                            sec_per_epoch=np.mean([h['time_s'] for h in hist]), cm=cm, per_cls=per_cls))
        print(f"==> {run}: test acc {acc:.4f}  macro-F1 {f1:.4f}\n")
        if device.type == 'cuda': torch.cuda.empty_cache()

res = pd.DataFrame(results).drop(columns=['cm', 'per_cls']).sort_values('test_acc', ascending=False)
res.to_csv(os.path.join(save_root, 'results.csv'), index=False)
display(res)
# mean ± std over seeds (robustness check, Table 2 in the report)
seed_tab = res.groupby('experiment')['test_acc'].agg(['count', 'mean', 'std'])
display(seed_tab[seed_tab['count'] > 1])

## 5. Plots

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for name, h in histories.items():
    if '_s' not in name:   # seed-0 runs only
        ax[0].plot(h['epoch'], h['val_acc'], label=name)
ax[0].set_xlabel('epoch'); ax[0].set_ylabel('validation accuracy'); ax[0].legend(fontsize=7)
r = res[res['seed'] == 0].sort_values('test_acc')
ax[1].barh(r['name'], r['test_acc']); ax[1].set_xlabel('test accuracy (seed 0)')
plt.tight_layout(); plt.show()

best = max(results, key=lambda d: d['test_acc'])
cmn = best['cm'] / best['cm'].sum(1, keepdims=True)
plt.figure(figsize=(5, 4.5)); plt.imshow(cmn, cmap='Blues')
plt.xticks(range(10), CLASSES, rotation=60); plt.yticks(range(10), CLASSES)
plt.title(f"Confusion matrix (row %) – {best['name']}"); plt.colorbar(); plt.tight_layout(); plt.show()

## 6. Note on the transfer-learning model (EffNet_clf)
`EffNet_clf.forward` applies the ImageNet `Normalize(mean, std)` directly to 0–255 pixels.
`Normalize` expects 0–1 input, so add `rgb = rgb / 255` before `self.rgb_normalizer(rgb)`.
EfficientNet also shrinks a 32×32 image to 1×1 before pooling; resizing the input (e.g. to 224×224) usually helps a pretrained backbone.